# <font color='red'>
---
## <center> <font color='red'> Masters in Mathematical Finance
#### <center> <font color='red'> 2024 / 26
# <center> <font color='red'> Masters' Final Work
---
# <center> <font color='red'><font> Student: Petr Terletskiy </font>
### <center> <font color='red'><font> Number: l63023 </font>
---
##### <center>  <font color='red'><font> BTC Daily Direction Prediction Using KANs Within the AFML Framework</font>

---

This notebook implements the full experimental pipeline for a Mathematical Finance Masters thesis at ISEG. 

The objective is to evaluate whether Kolmogorov–Arnold Networks (KANs) can predict Bitcoin daily price direction, benchmarked against AR Logistic, Logistic Regression, Random Forest, XGBoost, and LSTM models. 

Evaluation follows López de Prado's *Advances in Financial Machine Learning* (2018) framework: triple-barrier labeling, CUSUM event filtering, sample uniqueness weighting, and Combinatorial Purged Cross-Validation (CPCV, N=8, k=2) with purging and embargo to prevent information leakage. Hyperparameter tuning is performed per split via Optuna TPE with nested Purged K-Fold ensuring DSR and PBO remain valid. Post-CPCV analysis includes the Deflated Sharpe Ratio, Probability of Backtest Overfitting, DeLong pairwise AUC significance tests, and KAN symbolic extraction following the VIX KAN paper's Algorithm 1. Raw OHLCV data spans November 2014 to May 2026 to provide the 252-day lookback required by the longest-warmup features; the CUSUM event filter is applied from August 2015 onward, the date of Ethereum's Frontier launch and the first day with valid ETH/USD price data needed for the eth_btc_ratio feature. The feature set contains 62 columns total entering MDA selection: 25 technical, 9 mathematical, 22 external (13 macro, 1 crypto-macro, 8 on-chain), and 6 autoregressive lag features. AR Logistic uses lag features as its sole input set; the other five models receive them through the same MDA-selected feature pool as the engineered features.

**Code structure:**
- `src/pre_cpcv/` (data loading, labeling, sample weights, feature engineering, alignment)
- `src/cpcv/` (cross-validation splits, preprocessing, per-split hyperparameter tuning, models, calibration, pipeline orchestration)
- `src/post_cpcv/` (evaluation, symbolic extraction)

# 1. Dependencies

## 1.1. Installations

In [ ]:
%pip install yfinance coinmetrics.api_client matplotlib numpy pandas pandas-datareader statsmodels scikit-learn pyarrow optuna xgboost torch pykan pyyaml ipywidgets --quiet 
%pip install git+https://github.com/Blealtan/efficient-kan.git --quiet

## 1.2. Libraries & Project Modules

In [ ]:
%load_ext autoreload
%autoreload 2

# ── stdlib & third-party ──────────────────────────────────────────────
import os
import random
import torch
import logging
import sympy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, roc_auc_score,
                             ConfusionMatrixDisplay, RocCurveDisplay,)

# ── project modules ───────────────────────────────────────────────────
# Pre-CPCV
from src.pre_cpcv.data_loader import load_btc_daily
from src.pre_cpcv.labeling import compute_daily_volatility, cusum_filter, triple_barrier_labels, drop_rare_labels, run_labeling_pipeline
from src.pre_cpcv.sample_weights import compute_sample_weights
from src.pre_cpcv.features import compute_ta_features, compute_math_features, compute_lag_features, apply_log_transforms, build_feature_matrix
from src.pre_cpcv.external_features import build_external_features
from src.pre_cpcv.pre_cpcv_plots import (plot_cusum_filter, plot_tbl_examples, plot_label_distribution, plot_feature_distributions,
                                plot_feature_correlation, plot_feature_label_mutual_info, plot_adf_stationarity)
from src.pre_cpcv.alignment import align_for_cv, validate_alignment

# CPCV
from src.cpcv.cv import generate_cpcv_splits, build_path_matrix, get_split_info
from src.cpcv.cpcv_plots import pick_demo_splits, plot_btc_with_groups, plot_train_test_timelines, print_purge_embargo_detail, audit_cpcv_leakage
from src.cpcv.preprocessing import preprocess_fold
from src.cpcv.models import create_model, list_models
from src.cpcv.calibration import Calibrator
from src.cpcv.pipeline import run_cpcv_pipeline

# Post-CPCV
from src.post_cpcv.evaluation import analyze_results, compute_pbo
from src.post_cpcv.path_explorer import plot_paths_for_model, plot_paths_grid, interactive_path_explorer
from src.post_cpcv.diagnostics import (build_path_dispersion_table, summarize_path_dispersion, render_sharpe_distribution,
                                       render_regime_concentration_scatter, compute_bet_size_summary, collect_bet_sizes,
                                       render_bet_size_histograms, calibration_mean_audit, compute_reliability_curve,
                                       render_reliability_diagrams, render_feature_stability, print_feature_stability_table,
                                       render_confusion_matrices)
from src.post_cpcv.symbolic_extraction import run_symbolic_extraction

# ── logging setup ─────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",)

# ── Reproducibility: global seeds ─────────────────────────────────────
# Setting these at notebook level ensures all stochastic components of
# the pipeline (numpy RNG, PyTorch RNG on CPU and CUDA, Python random,
# Python hash randomization, PyTorch deterministic algorithms) start
# from the same state across runs. The Optuna TPE sampler seed is set
# separately inside tuning.py.

GLOBAL_SEED = 42

# Layer 2: global seeds for python, numpy, torch
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

# Layer 3: PyTorch deterministic algorithms (slower, but reproducible)
# Enable for locked thesis run; disable during development if the
# slowdown becomes painful.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Global seed: {GLOBAL_SEED}")
print(f"PyTorch deterministic algorithms: enabled")
print(f"CUDA available: {torch.cuda.is_available()}")

print("\n\n✅ Libraries imported successfully!")

# 2. Data Engineering

## 2.1. BTC Daily OHLCV Data

In [ ]:
START_DATE, END_DATE = '2014-11-01', '2026-05-01'

df_raw = load_btc_daily('BTC-USD', start=START_DATE, end=END_DATE)
df_raw.head()

## 2.2. Labeling

### Daily Volatility

In [ ]:
# ── Step 1: Daily volatility ──────────────────────────────────────────
daily_vol = compute_daily_volatility(df_raw["Close"], span=50)
daily_vol.plot(title="Daily Volatility (EWMA span=50)", figsize=(12, 3))

### CUSUM filter

In [ ]:
# ── Step 2: CUSUM filter ──────────────────────────────────────────────
log_returns = np.log(df_raw["Close"] / df_raw["Close"].shift(1)).dropna()

CUSUM_MULT = 1.00
CUSUM_START_DATE = "2015-08-08"   # first day with valid ETH/BTC ratio

h = CUSUM_MULT * daily_vol.mean()
t_events = cusum_filter(log_returns, h)

n_pre_filter = len(t_events)
t_events = t_events[t_events >= pd.Timestamp(CUSUM_START_DATE)]
n_dropped = n_pre_filter - len(t_events)

print(f"CUSUM threshold: {h:.6f}")
print(f"Total events fired: {n_pre_filter}")
print(f"Events before {CUSUM_START_DATE}: {n_dropped} (dropped)")
print(f"Events in analysis window: {len(t_events)}")

In [ ]:
# CUSUM filter visualization
fig = plot_cusum_filter(log_returns=log_returns, t_events=t_events,
                        h=h, zoom_start="2026-01-01", zoom_end="2026-04-01")
plt.show()

### Triple-Barrier Labeling

In [ ]:
# ── Step 3: Triple-barrier labeling ───────────────────────────────────
PT_SL_COMBO, NUM_DAYS, MIN_RETURN = (1.5, 1.5), 10, 0.02

bins = triple_barrier_labels(df_raw["Close"], t_events, trgt=daily_vol,
                             pt_sl=PT_SL_COMBO, num_days=NUM_DAYS, min_return=MIN_RETURN)

holding_days = (bins["t1"] - bins.index).dt.days
print(f"\nHolding period: mean={holding_days.mean():.1f}, median={holding_days.median():.1f} days")
print(f"Hit vertical barrier: {(holding_days >= NUM_DAYS).mean():.1%}")

In [ ]:
# Triple-barrier labeling examples
fig = plot_tbl_examples(bins=bins, close=df_raw["Close"], daily_vol=daily_vol,
                        pt_sl=PT_SL_COMBO, num_days=NUM_DAYS,
                        zoom_start="2026-01-01", zoom_end="2026-04-01")
plt.show()

### Drop rare labels

In [ ]:
# ── Step 4: Drop rare labels ─────────────────────────────────────────
bins = drop_rare_labels(bins, min_pct=0.085)
print(f"{len(bins)} labels | classes: {bins['bin'].value_counts().to_dict()}")
bins.head(10)

### Label Distribution

In [ ]:
# Label distribution
fig = plot_label_distribution(bins=bins)
plt.show()

## 2.3. Sample Weights

In [ ]:
# ── Step 5: Sample weights ────────────────────────────────────────────
sample_w = compute_sample_weights(bins, df_raw.index, time_decay_factor=0.4,
                                  weight_cap_quantile=0.99) # weight_cap_quantile = 1.00 to disable capping
sample_w.describe()

In [ ]:
# ── Visualize sample weights ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(sample_w, bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(sample_w.mean(), color="red", ls="--", label=f"mean={sample_w.mean():.2f}")
axes[0].set_title("Sample Weight Distribution")
axes[0].set_xlabel("Weight")
axes[0].legend()

axes[1].plot(sample_w.index, sample_w.values, lw=0.6)
axes[1].set_title("Sample Weights Over Time")
axes[1].set_ylabel("Weight")

plt.tight_layout()
plt.show()

In [ ]:
# ── Heavy-weight events (diagnostic) ─────────────────────────────────
heavy_threshold = sample_w.quantile(0.99)
heavy = sample_w[sample_w >= heavy_threshold].sort_values(ascending=False)
heavy_df = pd.DataFrame({
    "weight": heavy,
    "label": bins.loc[heavy.index, "bin"],
    "return": bins.loc[heavy.index, "ret"],
    "holding_days": (bins.loc[heavy.index, "t1"] - heavy.index).dt.days,
})
print(f"Events at or above 99th percentile ({heavy_threshold:.2f}): {len(heavy_df)}\n")
print(heavy_df.to_string())

## 2.4. Feature Engineering

In [ ]:
# ── Lag features (precomputed log-return lags for AR Logistic) ───────
lag_features = compute_lag_features(df_raw)
print(f"Lag features: {list(lag_features.columns)}")
print(f"Shape: {lag_features.shape}")
lag_features.tail(10)

In [ ]:
# ── Technical Analysis features ──────────────────────────────
ta_features = compute_ta_features(df_raw)
print(f"TA features: {list(ta_features.columns)}")
print(f"Shape: {ta_features.shape}")
print(f"NaN rows (any): {ta_features.isna().any(axis=1).sum()}")
ta_features.tail(10)

In [ ]:
# ── External features (macro + crypto) ───────────────────────────────
external = build_external_features(df_raw)
print(f"External features: {list(external.columns)}")
print(f"Shape: {external.shape}")
external.tail(10)

In [ ]:
# ── Mathematical features (AFML Part 4) ─────────────────────
# full run (~40 min, cached after first run)
math_features = compute_math_features(df_raw, which="all")
print(f"Math features: {list(math_features.columns)}")
print(f"Shape: {math_features.shape}")
print(f"NaN rows (any): {math_features.isna().any(axis=1).sum()}")
math_features.tail(10)

In [ ]:
# ── Assemble feature matrix & Log transforms ─────────────────
feature_matrix = pd.concat([ta_features, math_features, external, lag_features], axis=1)
feature_matrix = apply_log_transforms(feature_matrix)

# drop columns with >50% NaN
nan_pct = feature_matrix.isna().mean()
high_nan = nan_pct[nan_pct > 0.50].index.tolist()
if high_nan:
    feature_matrix = feature_matrix.drop(columns=high_nan)
    print(f"⚠ Dropped {len(high_nan)} column(s) with >50% NaN: {high_nan}")

print(f"Final feature matrix: {feature_matrix.shape[1]} features, {feature_matrix.shape[0]} rows")
print(f"Columns: {list(feature_matrix.columns)}")
print(f"NaN rows (any): {feature_matrix.isna().any(axis=1).sum()}")
feature_matrix.describe()

## 2.5. Exploratory Data Analysis

### Feature Distributions

In [ ]:
# Feature distributions histogram grid
fig = plot_feature_distributions(feature_matrix=feature_matrix,
                                 n_cols=4, kurtosis_threshold=10.0)
plt.show()

### Feature Correlation

In [ ]:
# Feature correlation matrix
fig = plot_feature_correlation(feature_matrix=feature_matrix,
                               corr_threshold=0.9, annotate=True)
plt.show()

### Stationarity ADF test

In [ ]:
# ADF stationarity test
fig, adf_df = plot_adf_stationarity(feature_matrix=feature_matrix,
                                    significance=0.05)
plt.show()

# FFD column list (manual choice based on stationarity diagnostic above)
COLUMNS_TO_FFD = ['atr']
print(f"\nColumns for FFD: {COLUMNS_TO_FFD}")

### Mutual Infomation - Features VS Label 

In [ ]:
# Feature-label mutual information
fig = plot_feature_label_mutual_info(feature_matrix=feature_matrix, bins=bins,
                                     mi_threshold=1e-6, seed=42)
plt.show()

## 2.6. Data Alignment

In [ ]:
# ── Alignment ────────────────────────────────────────────────
X, y, w, t1 = align_for_cv(feature_matrix, bins, sample_w)
validate_alignment(X, y, w, t1)
X.head()

# 3. Cross-Validation Framework

## 3.1. CPCV Split Generation

In [ ]:
# ── CPCV: Split generation ────────────────────────────────────────────
N_GROUPS, K_TEST, EMBARGO_PCT = 8, 2, 0.01

splits = generate_cpcv_splits(X, t1, n_groups=N_GROUPS, k=K_TEST, embargo_pct=EMBARGO_PCT)
n_paths, path_map = build_path_matrix(n_groups=N_GROUPS, k=K_TEST)
split_info = get_split_info(X, t1, n_groups=N_GROUPS, k=K_TEST, embargo_pct=EMBARGO_PCT,
                            splits=splits, path_map=path_map, n_paths=n_paths)

## 3.2. CPCV EDA

### Partition Overview

In [ ]:
# Group partition overview
fig = plot_btc_with_groups(X=X, df_raw=df_raw, n_groups=N_GROUPS,
                           use_log_scale=True) # set False for linear-scale view
plt.show()

### Train / Test Splits

In [ ]:
# Train/test timelines for three representative splits
fig = plot_train_test_timelines(X=X, splits=splits, n_groups=N_GROUPS,
                                k=K_TEST) # demo_splits=[0, 14, 27] for specific splits
plt.show()

### Purging & Embargo Verification

In [ ]:
# Purging and embargo detail (text dump)
print_purge_embargo_detail(X=X, t1=t1, splits=splits, n_groups=N_GROUPS, k=K_TEST)

### Leakage Verification

In [ ]:
# CPCV leakage audit (all splits)
audit_df = audit_cpcv_leakage(X=X, t1=t1, splits=splits, n_groups=N_GROUPS,
                              k=K_TEST, split_info=split_info)

print(audit_df[audit_df["leaks"] > 0])

# 4. Model Training

## 4.1. Benchmark Model

In [ ]:
# ── Econometric baseline (AR Logistic, 3 seeds, ~10 secs) ──────────────
ar_logistic_results = run_cpcv_pipeline(X, y, w, t1, bins_ret=bins["ret"],
                      models=["ar_logistic"], n_seeds=3, ffd_columns=COLUMNS_TO_FFD,
                      splits=splits, path_map=path_map, n_paths=n_paths, split_info=split_info)

## 4.2. Logistic Regression

In [ ]:
# ── Fraction of Top Features surviving MDA ──────────────────
TOP_K_FRAC = 0.25

In [ ]:
# ── Logistic Regression (3 seeds, ~15 min) ────────────────────────────────────
lr_results = run_cpcv_pipeline(
             X, y, w, t1, bins_ret=bins["ret"],
             models=["logistic"], n_seeds=3, ffd_columns=COLUMNS_TO_FFD,
             top_k_frac=TOP_K_FRAC, tune=True, tune_models=["logistic"], n_trials=30,
             splits=splits, path_map=path_map, n_paths=n_paths, split_info=split_info)

## 4.3. Ensemble Models

In [ ]:
# ── Ensemble Models (3 seeds, ~55 min) ────────────────────────────────────
ensembles_results = run_cpcv_pipeline(
                    X, y, w, t1, bins_ret=bins["ret"],
                    models=["random_forest", "xgboost"], n_seeds=3, ffd_columns=COLUMNS_TO_FFD,
                    top_k_frac=TOP_K_FRAC, tune=True, tune_models=["random_forest", "xgboost"], n_trials=30,
                    splits=splits, path_map=path_map, n_paths=n_paths, split_info=split_info)

## 4.4. LSTM

In [ ]:
# ── LSTM (2 seeds, ~35 min) ────────────────────────────────────
lstm_results = run_cpcv_pipeline(
               X, y, w, t1, bins_ret=bins["ret"],
               models=["lstm"], n_seeds=2, ffd_columns=COLUMNS_TO_FFD,
               top_k_frac=TOP_K_FRAC, tune=True, tune_models=["lstm"], n_trials=30,
               splits=splits, path_map=path_map, n_paths=n_paths, split_info=split_info)

## 4.5. KAN

In [ ]:
# ── KAN (2 seeds, ~40 min) ────────────────────────────────────
kan_results = run_cpcv_pipeline(
              X, y, w, t1, bins_ret=bins["ret"],
              models=["kan"], n_seeds=2, ffd_columns=COLUMNS_TO_FFD,
              top_k_frac=TOP_K_FRAC, tune=True, tune_models=["kan"], n_trials=30,
              splits=splits, path_map=path_map, n_paths=n_paths, split_info=split_info)

## 4.6. Merge Results

In [ ]:
# ── Merge results ────────────────────────────────────────────────────
all_predictions = {
    **ar_logistic_results["predictions"],
    **lr_results["predictions"],
    **ensembles_results["predictions"],
    **lstm_results["predictions"],
    **kan_results["predictions"],
}

results = {
    "predictions": all_predictions,
    "split_info": lr_results["split_info"],
    "path_map": lr_results["path_map"],
    "n_paths": lr_results["n_paths"],
    "n_splits": lr_results["n_splits"],
    "n_groups": N_GROUPS,
    "models": ["ar_logistic", "logistic", "random_forest", "xgboost", "lstm", "kan"],
    "n_seeds": 3,
}

print(f"Merged: {len(all_predictions)} total prediction entries")
print(f"Models: {results['models']}")

# 5. Post-CPCV Evaluation

## 5.1. Predictive Performance

### Model Comparison

In [ ]:
analysis = analyze_results(results)

### Confusion Matrices

In [ ]:
fig = render_confusion_matrices(results, seed_mode="average") # seed_mode= "average", "best", "seed_0", "seed_1", "seed_2"  
plt.show()

## 5.2. Financial Performance

### Equity Curves

In [ ]:
interactive_path_explorer(results, analysis)

In [ ]:
plot_paths_grid(results, analysis)

### Sharpe Distribution

In [ ]:
fig = render_sharpe_distribution(analysis)
plt.show()

### Path Dispersion & Regime Concentration

In [ ]:
dispersion = build_path_dispersion_table(analysis, k=5)
display(dispersion.style.format({"sharpe": "{:.3f}", "cum_return": "{:.3f}",
                                 "max_dd": "{:.3f}", "top_k_share": "{:.1%}"}))

In [ ]:
summary = summarize_path_dispersion(dispersion)
display(summary.style.format({"sharpe_min": "{:.3f}", "sharpe_median": "{:.3f}", "sharpe_max": "{:.3f}",
                              "cum_min": "{:.3f}", "cum_median": "{:.3f}", "cum_max": "{:.3f}", "avg_top_k_share": "{:.1%}"}))

In [ ]:
dispersion = build_path_dispersion_table(analysis, k=5)
fig = render_regime_concentration_scatter(dispersion, k=5)
plt.show()

## 5.3. Trading Behaviour

### Bet-Size Distribution

In [ ]:
bet_summary = compute_bet_size_summary(analysis)
display(bet_summary.style.format({"n_events": "{:,}", "abstention_rate": "{:.1%}",
                                  "mean_abs_bet": "{:.3f}", "median_abs_bet": "{:.3f}",
                                  "share_at_max": "{:.1%}", "long_share": "{:.1%}",
                                  "short_share": "{:.1%}"}))

In [ ]:
fig = render_bet_size_histograms(analysis)
plt.show()

## 5.4. Statistical Robustness

### FFD Stability, PBO & DSR

In [ ]:
# ── FFD d* values across folds ───────────────────────────────────────
ffd_stab = analysis["ffd_stability"]
for col, vals in ffd_stab["d_star_by_column"].items():
    print(f"{col}: mean d*={ffd_stab['mean_d_star'][col]:.3f}, "
          f"std={ffd_stab['std_d_star'][col]:.3f}, values={vals}")

# ── Probability of Backtest Overfitting ──────────────────────────────
print(f"\n\nPBO: {analysis['pbo']:.4f}")
if analysis["pbo"] < 0.3:
    print("  Model selection appears robust (PBO < 0.3).")
elif analysis["pbo"] > 0.5:
    print("  Warning: in-sample winner tends to underperform OOS (PBO > 0.5).")
else:
    print("  Moderate overfitting risk (0.3 < PBO < 0.5).")

# DSR per model
print("\n\nDeflated Sharpe Ratios:")
for s in analysis["all_summaries"]:
    dsr_flag = " ✓" if s["dsr"] > 0.95 else ""
    print(f"  {s['model_name']:>20s}: DSR={s['dsr']:.4f}{dsr_flag}")

## 5.5. Methodology Audits

### Calibration Audit

In [ ]:
audit_df = calibration_mean_audit(results, y)

In [ ]:
fig = render_reliability_diagrams(results, analysis)
plt.show()

### Feature Sensibility

In [ ]:
fig = render_feature_stability(analysis["feature_stability"])
plt.show()

ranking = print_feature_stability_table(analysis["feature_stability"],
                                        all_features=list(X.columns),
                                        threshold_pct=50)

In [ ]:
# Build a sub-analysis dropping LSTM
sub_path_results = {
    m: paths for m, paths in analysis["path_results"].items() if m != "lstm"
}
sub_path_sharpes = analysis["path_sharpes"][[i for i, m in enumerate(results["models"]) if m != "lstm"], :]

# Recompute PBO on the five-model subset
pbo_5model = compute_pbo(sub_path_sharpes)
print(f"Six-model PBO: {analysis['pbo']:.3f}")
print(f"Five-model PBO (LSTM excluded): {pbo_5model:.3f}")

# 6. Symbolic Extraction

## 6.1. Run Extraction

In [ ]:
symbolic = run_symbolic_extraction(cpcv_results=results, X=X, y=y, w=w, t1=t1,
                                   n_top_features=5, fold_selection="last",
                                   use_multkan=False)

## 6.2. Extracted Formula

In [ ]:
print("Decision function:")
print(f"  {symbolic['decision_function']}")
print(f"\nP(up) = {symbolic['p_up_formula']}")
print(f"\nSurviving features: {symbolic['surviving_features']}")

## 6.3. Predictive Performance Retention

In [ ]:
print(f"Pre-symbolic accuracy:  {symbolic['pre_symbolic_accuracy']:.4f}")
print(f"Post-symbolic accuracy: {symbolic['post_symbolic_accuracy']:.4f}")
print(f"Symbolification rate:   {symbolic.get('symbolification_rate', 'N/A')}")
print(f"Pruned architecture:    {symbolic.get('pruned_architecture', 'N/A')}")

## 6.4. Feature Sensivity

### Symbolic Derivatives

In [ ]:
decision_expr = symbolic["sympy_objects"]["decision"]
features = symbolic["surviving_features"]

if decision_expr is not None and decision_expr != "extraction_failed":
    print("Partial derivatives (symbolic form):\n")
    for feat in features:
        sensitivity = sympy.diff(decision_expr, sympy.Symbol(feat))
        print(f"  ∂(decision)/∂({feat}) =")
        print(f"    {sensitivity}\n")

### Numerical sensitivity at the dataset mean

In [ ]:
sym_vars = {f: sympy.Symbol(f) for f in features}
X_features = X[features]
means = X_features.mean()
stds = X_features.std()

sensitivity_rows = []
for feat in features:
    deriv = sympy.diff(decision_expr, sym_vars[feat])
    deriv_at_mean = float(deriv.subs({sym_vars[f]: means[f] for f in features}))
    sigma_effect = deriv_at_mean * stds[feat]
    approx_delta_p = sigma_effect / 4.0   # sigmoid slope at p=0.5

    sensitivity_rows.append({
        "feature": feat,
        "mean_value": means[feat],
        "std_value": stds[feat],
        "d_decision/d_feature_at_mean": deriv_at_mean,
        "sigma_effect_on_decision": sigma_effect,
        "approx_sigma_delta_p": approx_delta_p,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows).set_index("feature")
print("Feature sensitivity at the dataset mean:\n")
print(sensitivity_df.to_string(float_format=lambda x: f"{x:+.4f}"))

### Marginal Effect Plots

In [ ]:
decision_fn = sympy.lambdify(
    [sym_vars[f] for f in features],
    decision_expr, modules="numpy")

n_feat = len(features)
fig, axes = plt.subplots(1, n_feat, figsize=(4.5 * n_feat, 4), sharey=True)
if n_feat == 1:
    axes = [axes]

for ax, feat in zip(axes, features):
    sweep = np.linspace(
        X_features[feat].quantile(0.05),
        X_features[feat].quantile(0.95),
        100)
    
    fixed = {f: X_features[f].median() for f in features}

    p_up = []
    for v in sweep:
        fixed[feat] = v
        d = decision_fn(*[fixed[f] for f in features])
        p_up.append(1.0 / (1.0 + np.exp(-d)))

    ax.plot(sweep, p_up, linewidth=2)
    ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5,
               label="P=0.5 (abstention)")
    ax.axvline(X_features[feat].median(), color="red", linestyle=":",
               alpha=0.5, label="Median")
    ax.set_xlabel(feat)
    ax.set_title(f"Marginal effect: {feat}")
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)

axes[0].set_ylabel("P(up)")
axes[0].legend(loc="best", fontsize=9)
fig.suptitle(
    "Marginal effect of each feature on P(up)\n"
    "(other features held at their median)",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.show()

### Term-structure summary

In [ ]:
formula_str = str(decision_expr)
term_counts = {feat: formula_str.count(feat) for feat in features}

summary = sensitivity_df.copy()
summary["n_terms_in_formula"] = pd.Series(term_counts)
summary = summary[[
    "mean_value", "std_value", "n_terms_in_formula",
    "d_decision/d_feature_at_mean", "sigma_effect_on_decision",
    "approx_sigma_delta_p",
]]

print("Formula term-structure summary:\n")
print(summary.to_string(float_format=lambda x: f"{x:+.4f}"))